In [2]:
import polars as pl
import pandas as pd
import duckdb

In [10]:
import os
import glob
import pandas as pd

# Root folder that contains subfolders of CSV files
root_dir = r"C:/Users/ekadw/Documents/DATA/Stock_Market/"

# Recursively find all CSV files
files = glob.glob(os.path.join(root_dir, "**", "*.csv"), recursive=True)

print(f"Found {len(files)} CSV files")

for f in files:
    try:
        # Read CSV
        df = pd.read_csv(f)
        
        # Build matching parquet path (same folder, same name, .parquet extension)
        parquet_path = os.path.splitext(f)[0] + ".parquet"
        
        # Save to Parquet
        df.to_parquet(parquet_path, index=False)
        
        #print(f"Converted: {f} -> {parquet_path}") # it is too long
    except Exception as e:
        print(f"Failed to convert {f}: {e}")


Found 8050 CSV files


In [28]:
import duckdb
con = duckdb.connect()

query = """
WITH CTE_filename AS (
SELECT
    *,
    "Adj Close" AS Adj_Close,
    regexp_extract(filename, '([^/\\\\]+)\\.parquet$') AS source,
    regexp_extract(filename, '([^/\\\\]+)\\.parquet$', 1) AS ticker,
    EXTRACT(YEAR FROM CAST(Date AS DATE)) AS year,
    LAG(Close, 1) OVER (PARTITION BY ticker ORDER BY Date ASC) AS Close_Lag_1,
    LAG(Adj_Close,1) OVER (PARTITION BY ticker ORDER BY Date ASC) AS Adj_Close_Lag_1,
    High - Low AS hi_lo_range
FROM read_parquet('C:/Users/ekadw/Documents/DATA/Stock_Market/*/*.parquet', filename=true)
), CTE_lag AS (
SELECT
    *,
    (Close/Close_Lag_1) - 1 AS returns
FROM CTE_filename
), CTE_returns AS (
SELECT
    *,
    AVG(returns) OVER (PARTITION BY ticker ORDER BY Date ROWS BETWEEN 9 PRECEDING AND CURRENT ROW) AS ma10_returns
FROM CTE_lag
)
SELECT
    Date,
    Open,
    High,
    Low,
    Close,
    Adj_Close,
    Volume,
    source,
    ticker,
    year,
    Close_Lag_1,
    Adj_Close_Lag_1,
    returns,
    hi_lo_range,
    ma10_returns
FROM CTE_returns
WHERE ticker == 'AAON'
LIMIT 100
"""
stock_market_A = con.execute(query).fetchdf()
stock_market_A.head()

OutOfMemoryException: Out of Memory Error: Allocation failure